# Week 6, Day 5 — The Trading Floor, Live
### Local Models Edition — by Abhishek — Capstone

The whole course lands here: `backend/trading_floor.py` runs all four
traders on a schedule, and `demo/ui.py` (via `app.py`) shows them live in a
Gradio dashboard — portfolio value, chart, log, holdings, and transactions
per trader, refreshing on a timer. This is the **real dashboard from the
original course**, completely unmodified — it only ever reads `accounts.db`
and the log table, so it doesn't care whether the traders behind it are
running on GPT-5 or a free local Llama.

**What's different from the original for this edition:**
`trading_floor.py`'s `model_names` defaults to local Ollama models instead
of paid frontier models (see Day 4). Everything downstream — the scheduler,
the tracer, the dashboard — is untouched.


## 0. Setup

In [ ]:
%pip install -q openai-agents mcp duckduckgo-search python-dotenv gradio plotly pandas fastapi uvicorn


## Running one round by hand, before scheduling it

`create_traders()` builds the four `Trader` objects from `names`,
`lastnames`, and `model_names` in `backend/trading_floor.py`.


In [ ]:
from backend.reset import reset_traders
reset_traders()

from backend.trading_floor import create_traders
from backend.tracers import LogTracer
from agents import add_trace_processor
import asyncio

add_trace_processor(LogTracer())
traders = create_traders()
for t in traders:
    print(t.name, t.lastname, t.model_name)


In [ ]:
await asyncio.gather(*[t.run() for t in traders])
print("One round complete for all four traders.")


## The scheduler

`run_every_n_minutes()` in `backend/trading_floor.py` loops forever,
running all four traders together and then sleeping. In a notebook we run
it as a background task so we can still use the notebook while it works.


In [ ]:
from backend.trading_floor import run_every_n_minutes

scheduler_task = asyncio.create_task(run_every_n_minutes())
print("Scheduler started in the background. Cancel with: scheduler_task.cancel()")


## The dashboard

`app.py` launches `demo.ui.create_ui()` — four `TraderView` panels, each
polling its `Trader`'s underlying `Account` on a timer. Exactly the
original course's UI. Run it from a terminal with `python app.py` for the
full experience (it opens a browser tab), or launch inline here.


In [ ]:
from demo.ui import create_ui
from demo.util import css, js

ui = create_ui()
ui.launch(css=css, js=js)


## The optional HTTP API and a separate frontend

`backend/api.py` serves the same account/log data as JSON
(`/api/traders`, `/api/traders/{name}`, `/api/traders/{name}/logs`) for a
fully decoupled frontend — the original course pairs this with a Vite/React
app in `frontend/`. That TypeScript frontend isn't reproduced in this
Python-notebook edition, but `api.py` is included unmodified: run it with

```bash
uv run uvicorn backend.api:app --port 8000
```

and it's ready for a frontend of your own — this is a natural stretch
project for stronger students.

## Congratulations

You've rebuilt the whole course end to end on free, local models: the
building blocks, LangGraph, `create_agent`, Deep Agents, the Sidekick, and
now the real multi-agent MCP trading floor with its own dashboard — running
the original course's actual `backend/` code, adapted at exactly one
function (`get_model()`) to reach a local model instead of a paid one.

## Exercise
1. Let the scheduler run for a few rounds and watch the dashboard update.
2. Set `USE_MANY_MODELS=true` in your `.env` and compare how four different
   free local models trade the same strategies.
3. **Stretch:** build a small frontend against `backend/api.py` — even a
   single HTML page with `fetch()` calls is enough to complete the original
   course's decoupled-architecture story.
